# Capture velocity, cross section, and loading rate

This notebook samples full-sphere incident direction discs. Each impact point gets a deterministic capture-velocity search using the Section-12 force and RK4. The cross section is the direction-averaged projected disc area multiplied by the captured fraction; there is no octant or 4π multiplicity factor. Timeouts fail closed instead of being labeled escaped. The loading quadrature uses the project's fixed reference vapor-speed distribution and prefactor. Small defaults are for interactive inspection, not quantitative claims. A named run may be resumed only with exactly matching parameters.

In [ ]:
%matplotlib inline
from dataclasses import replace

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from pmot.magnetic_fields import default_anti_helmholtz_config
from pmot.mot_multilevel import (
    CaptureSearchConfig, default_multilevel_mot_config, multilevel_mot_paths,
    run_capture_loading_study,
)
OUTPUT = multilevel_mot_paths()['statistics'] / 'notebook_capture_loading'
FIGURES = multilevel_mot_paths()['figures'] / 'notebook_capture_loading'

In [ ]:
disc_count = widgets.IntSlider(description='direction discs', min=1, max=25, value=2, continuous_update=False)
points_per_disc = widgets.IntSlider(description='points/disc', min=1, max=25, value=3, continuous_update=False)
launch_radius = widgets.FloatSlider(description='launch r [mm]', min=5, max=30, step=.5, value=15, continuous_update=False)
disc_radius = widgets.FloatSlider(description='disc R [mm]', min=1, max=20, step=.5, value=8, continuous_update=False)
velocity_guess = widgets.FloatSlider(description='v guess [m/s]', min=.25, max=30, step=.25, value=5, continuous_update=False)
velocity_tolerance = widgets.FloatSlider(description='v tol [m/s]', min=.1, max=2, step=.05, value=.5, continuous_update=False)
max_time = widgets.FloatSlider(description='max T [ms]', min=5, max=200, step=5, value=50, continuous_update=False)
dt = widgets.FloatSlider(description='dt [us]', min=2.5, max=20, step=2.5, value=5, continuous_update=False)
gradient = widgets.FloatSlider(description='dBz/dz [G/cm]', min=1, max=30, step=.5, value=10, continuous_update=False)
detuning = widgets.FloatSlider(description='cool Δ [MHz]', min=-40, max=-1, step=.5, value=-15, continuous_update=False)
cooling_power = widgets.FloatSlider(description='cool P [mW]', min=.1, max=60, step=.1, value=27, continuous_update=False)
repump_power = widgets.FloatSlider(description='repump P [mW]', min=.01, max=2, step=.01, value=.1, continuous_update=False)
seed = widgets.BoundedIntText(description='seed', min=0, max=2_000_000_000, value=20260918)
workers = widgets.IntSlider(description='workers', min=1, max=8, value=1, continuous_update=False)
run_name = widgets.Text(description='run name', value='interactive_001')
resume = widgets.Checkbox(description='resume matching run', value=False, indent=False)
run = widgets.Button(description='Run capture/loading', button_style='primary')
out = widgets.Output()

def run_study(_=None):
    with out:
        out.clear_output(wait=True); plt.close('all')
        search = CaptureSearchConfig(
            radial_distance_m=launch_radius.value*1e-3,
            disc_radius_m=disc_radius.value*1e-3,
            disc_count=disc_count.value, points_per_disc=points_per_disc.value,
            initial_velocity_guess_m_per_s=velocity_guess.value,
            velocity_tolerance_m_per_s=velocity_tolerance.value,
            maximum_simulation_time_s=max_time.value*1e-3,
            time_step_s=dt.value*1e-6, seed=seed.value, worker_count=workers.value,
            analysis_s_bin_count=max(2, min(12, points_per_disc.value)),
        )
        config = replace(
            default_multilevel_mot_config(),
            cooling_detuning_rad_per_s=2*np.pi*detuning.value*1e6,
            cooling_power_w_per_beam=cooling_power.value*1e-3,
            repump_power_w_per_beam=repump_power.value*1e-3,
        )
        coil = default_anti_helmholtz_config(target_gradient_g_per_cm=gradient.value)
        name = run_name.value.strip()
        if not name or not all(char.isalnum() or char in '-_' for char in name):
            raise ValueError('run name must use only letters, digits, - or _')
        print(f'Running {search.disc_count*search.points_per_disc} capture boundaries. This can be slow.')
        result = run_capture_loading_study(
            search, config=config, coil_config=coil,
            output_directory=OUTPUT/name, figure_directory=FIGURES/name,
            resume=resume.value,
        )
        rows = [{
            'disc': s.disc_index, 'point': s.point_index, 'impact_mm': 1e3*s.s_m,
            'capture_velocity_m_per_s': s.capture_velocity_m_per_s,
            'resolution_m_per_s': s.velocity_resolution_m_per_s,
        } for s in result.samples]
        display(pd.DataFrame(rows))
        v = np.asarray([s.velocity_m_per_s for s in result.spectrum])
        sigma = 1e6*np.asarray([s.capture_cross_section_m2 for s in result.spectrum])
        fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
        ax.step(v, sigma, where='post'); ax.set(xlabel='speed [m/s]', ylabel='capture cross section [mm²]')
        ax.grid(alpha=.25); plt.show()
        print(f'Loading rate = {result.loading.loading_rate_atoms_per_s:.6g} atoms/s')
        print(f'Wall time = {result.wall_time_s:.3f} s')
        print('Outputs:'); [print(f'  {name}: {path}') for name, path in result.output_paths.items()]

run.on_click(run_study)
display(widgets.VBox([
    widgets.HBox([disc_count, points_per_disc, workers]),
    widgets.HBox([launch_radius, disc_radius, seed]),
    widgets.HBox([velocity_guess, velocity_tolerance, max_time, dt]),
    widgets.HBox([detuning, cooling_power, repump_power, gradient]),
    widgets.HBox([run_name, resume, run]), out,
]))